# Baseline en test con LM Studio

Evalúa el modelo sin entrenar sobre `datos/test.jsonl`: 20 árboles por nivel en los niveles 2 a 6, con 3 intentos por árbol (300 episodios).

**Si cambias un presupuesto:** cambia también `NOMBRE` y `RUTA_RESULTADOS`, para no mezclar corridas.

**Antes de empezar:** abre LM Studio, carga el modelo e inicia el servidor local. Abre este notebook desde la carpeta `seleccion_arbol` y selecciona el kernel del entorno de conda `tipico`, que tiene `pandas` y `numpy`.

**Si se interrumpe:** ejecuta todo de nuevo. Cada episodio se guarda al terminar, así que continúa donde quedó. Si cambias de modelo, cambia también `NOMBRE`, o sus resultados se mezclarían en el mismo archivo con ese nombre.

## Configuración

In [1]:
from modelo import ModeloLMStudio

NOMBRE = 'Qwen 2B sin entrenar'
MODELO_LM_STUDIO = 'qwen3.5-2b'
URL = 'http://localhost:1235/v1'
RUTA_DATOS = 'datos/test.jsonl'
RUTA_RESULTADOS = 'resultados/baseline_test_lmstudio.jsonl'
NIVELES = [2, 3, 4, 5, 6]
INTENTOS = 3  # El k de pass@k
TOKENS_PENSAMIENTO = 3400  # Al agotarlos, se cierra el razonamiento y el modelo debe responder
TOKENS_RESPUESTA = 200  # Tokens para escribir los nodos después de ese cierre

modelo = ModeloLMStudio(
    nombre=MODELO_LM_STUDIO,
    url_base=URL,
    tokens_pensamiento=TOKENS_PENSAMIENTO,
    tokens_respuesta=TOKENS_RESPUESTA,
)

## Comprobar la conexión

Si esta celda falla, revisa que el servidor de LM Studio esté iniciado.

In [2]:
import json
import urllib.request

with urllib.request.urlopen(f'{URL}/models', timeout=10) as respuesta:
    disponibles = [m['id'] for m in json.load(respuesta)['data']]

print('Modelos disponibles:', disponibles)
assert MODELO_LM_STUDIO in disponibles, f'{MODELO_LM_STUDIO} no está disponible en LM Studio'

Modelos disponibles: ['qwen3.5-2b', 'text-embedding-nomic-embed-text-v1.5']


## Evaluación

In [ ]:
from evaluacion import evaluar_dataset

evaluar_dataset(modelo, NOMBRE, RUTA_DATOS, RUTA_RESULTADOS, niveles=NIVELES, intentos=INTENTOS)

Qwen 2B sin entrenar nivel 2 ep test-2-9 intento 2: recompensa=0.438 puntaje=7 óptimo=10 112.8s 3404tok


## Baseline por nivel

Puedes ejecutar estas celdas en cualquier momento para ver lo avanzado. Si la evaluación corre en otro proceso, no vuelvas a ejecutar la celda de evaluación: se repetirían episodios.

Las columnas de proporción van de 0 a 1.

- **pass@1:** intentos que logran el óptimo.
- **pass@3:** árboles donde al menos uno de sus 3 intentos logra el óptimo.
- **recompensa:** media del episodio. Igualar el mejor atajo vale 0,5, y una respuesta inválida, -1.
- **optimos:** árboles donde el modelo logró el mejor puntaje posible.
- **optimos_atajo:** óptimos de la estrategia "sin vecinos", que no razona sobre costos. El modelo debería superarla para mostrar que optimiza de verdad.
- **supera_voraz:** veces que superó la regla simple.
- **supera_atajos:** veces que superó al mejor atajo.
- **invalidos:** respuestas sin un formato válido.
- **cortes:** episodios que agotaron el pensamiento, y cuya respuesta fue forzada.

In [ ]:
from evaluacion import resumir, resumir_pass

resumir_pass(RUTA_RESULTADOS, k=INTENTOS)

In [ ]:
resumir(RUTA_RESULTADOS)

## Baseline total

In [ ]:
resumir_pass(RUTA_RESULTADOS, k=INTENTOS, por=['modelo'])

In [ ]:
resumir(RUTA_RESULTADOS, por=['modelo'])